In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import BallTree
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from scipy.optimize import minimize

In [ ]:
# LOAD THE DATA
merged = pd.read_csv("nypd_merged_2025.csv")
deployment = pd.read_csv('nypd_deployment_2025.csv')

In [ ]:
deployment = pd.read_csv('nypd_deployment_2025.csv')
deployment.columns = deployment.columns.str.lower()
deployment['arrest_precinct'] = pd.to_numeric(deployment['command'], errors='coerce')
deployment.loc[deployment['command'] == 'Central Park', 'arrest_precinct'] = 22

In [ ]:
print(merged.columns.tolist())

NEED TO ALSO LOAD THE PRECINT DATASET SO WE CAN MAP GRID CELLS TO THE NEAREST PRECINT.. POTENTIAL CODE BELOW ONCE WE HAVE IT!

To estimate deployment influence, you must transform raw points into a Density Map.
- Define a Grid: Create a bounding box for NYC and divide it into cells 
- Nearest Precinct: Calculate the centroid of each cell and find the Euclidean distance to the nearest Precinct station.
- Map Deployment: Merge your parsed PDF data (officers per precinct) to each grid cell based on its assigned precinct.

In [ ]:
precinct_locations = merged.groupby('ARREST_PRECINCT').agg(
    precinct_lat=('Latitude', 'mean'),
    precinct_lon=('Longitude', 'mean'),
    officers_assigned=('Total', 'first')
).reset_index().rename(columns={'ARREST_PRECINCT': 'precinct'})

print(precinct_locations.head())
print("Shape:", precinct_locations.shape)

In [ ]:
# Create grid cells from latitude/longitude

#LAT LONG TO NUMERIC
merged['Latitude'] = pd.to_numeric(merged['Latitude'], errors='coerce')
merged['Longitude'] = pd.to_numeric(merged['Longitude'], errors='coerce')

#DROP BAD COORDINATES
merged = merged.dropna(subset=['Latitude', 'Longitude'])
merged = merged[(merged['Latitude'] != 0) & (merged['Longitude'] != 0)]

#GRID CELLS
merged['lat_grid'] = merged['Latitude'].round(3)
merged['lon_grid'] = merged['Longitude'].round(3)

#GRID ID
merged['grid_id'] = (
    merged['lat_grid'].astype(str) + "_" + merged['lon_grid'].astype(str)
)

print(merged[['Latitude', 'Longitude', 'lat_grid', 'lon_grid', 'grid_id']].head())
print("Unique grid cells:", merged['grid_id'].nunique())

In [ ]:
grid_points = merged[["grid_id", "lat_grid", "lon_grid"]].drop_duplicates().dropna()

precinct_coords = np.radians(precinct_locations[["precinct_lat", "precinct_lon"]])
grid_coords = np.radians(grid_points[["lat_grid", "lon_grid"]])

tree = BallTree(precinct_coords, metric="haversine")
dist, ind = tree.query(grid_coords, k=1)

grid_points["nearest_precinct"] = precinct_locations.iloc[ind.flatten()]["precinct"].values
grid_points["estimated_officers"] = precinct_locations.iloc[ind.flatten()]["officers_assigned"].values

print(grid_points.head())

In [ ]:
# Merge nearest precinct/officer info back onto each arrest row
merged = merged.merge(
    grid_points[['grid_id', 'nearest_precinct', 'estimated_officers']],
    on='grid_id',
    how='left'
)

# Aggregate arrests to daily grid level
daily_grid = merged.groupby(
    ['grid_id', 'ARREST_DATE']
).agg({
    'ARREST_KEY': 'count',
    'ARREST_PRECINCT': 'first',
    'ARREST_BORO': 'first',
    'nearest_precinct': 'first',
    'estimated_officers': 'first',
    'Total': 'first',
    '% of Command': 'first',
    'Bureau': 'first',
    'Viper': 'first'
}).reset_index()

# Rename arrest count
daily_grid = daily_grid.rename(columns={
    'ARREST_KEY': 'crime_count',
    'Total': 'total',
    '% of Command': '% of command',
    'Bureau': 'bureau',
    'Viper': 'viper'
})

print(daily_grid.head())
print("Shape:", daily_grid.shape)

In [ ]:
daily_grid = daily_grid[daily_grid['grid_id'] != '0.0_0.0']

daily_grid['ARREST_DATE'] = pd.to_datetime(daily_grid['ARREST_DATE'])
daily_grid['month'] = daily_grid['ARREST_DATE'].dt.month
daily_grid['day_of_week'] = daily_grid['ARREST_DATE'].dt.dayofweek
daily_grid['weekend'] = daily_grid['day_of_week'].isin([5, 6]).astype(int)
daily_grid['week_of_year'] = daily_grid['ARREST_DATE'].dt.isocalendar().week.astype(int)
daily_grid['year'] = daily_grid['ARREST_DATE'].dt.year

daily_grid = daily_grid.sort_values(['grid_id', 'ARREST_DATE'])
daily_grid['lag_1d']  = daily_grid.groupby('grid_id')['crime_count'].shift(1)
daily_grid['lag_7d']  = daily_grid.groupby('grid_id')['crime_count'].shift(7)
daily_grid['lag_30d'] = daily_grid.groupby('grid_id')['crime_count'].shift(30)
daily_grid['rolling_7d_mean']  = daily_grid.groupby('grid_id')['crime_count'].transform(lambda x: x.shift(1).rolling(7).mean())
daily_grid['rolling_30d_mean'] = daily_grid.groupby('grid_id')['crime_count'].transform(lambda x: x.shift(1).rolling(30).mean())

daily_grid = daily_grid.dropna(subset=['lag_1d', 'lag_7d', 'lag_30d', 'bureau'])

print("Shape:", daily_grid.shape)
print("\nDate range:", daily_grid['ARREST_DATE'].min(), "→", daily_grid['ARREST_DATE'].max())
print("\nMissing values:\n", daily_grid.isnull().sum())

In [ ]:
daily_grid = daily_grid.dropna(subset=['nearest_precinct'])
print("Final shape:", daily_grid.shape)

In [ ]:
# ── 1. encode categoricals ───────────────────────────────────────────────────
le_boro = LabelEncoder()
le_bureau = LabelEncoder()
daily_grid['boro_encoded'] = le_boro.fit_transform(daily_grid['ARREST_BORO'].astype(str))
daily_grid['precinct_encoded'] = LabelEncoder().fit_transform(daily_grid['nearest_precinct'].astype(str))
daily_grid['bureau_encoded'] = le_bureau.fit_transform(daily_grid['bureau'].astype(str))
deployment_features = base_features + ['estimated_officers', '% of command']

# ── 2. feature sets ──────────────────────────────────────────────────────────
base_features = [
    'month', 'day_of_week', 'weekend', 'week_of_year',
    'boro_encoded', 'bureau_encoded', 'precinct_encoded',
    'lag_1d', 'lag_7d', 'lag_30d',
    'rolling_7d_mean', 'rolling_30d_mean'
]


target = 'crime_count'

# ── 3. time-based train/test split ───────────────────────────────────────────
daily_grid = daily_grid.sort_values('ARREST_DATE')
cutoff = daily_grid['ARREST_DATE'].quantile(0.8)

train = daily_grid[daily_grid['ARREST_DATE'] <= cutoff]
test  = daily_grid[daily_grid['ARREST_DATE'] > cutoff]

X_train_base = train[base_features]
X_test_base  = test[base_features]
X_train_dep  = train[deployment_features]
X_test_dep   = test[deployment_features]
y_train      = train[target]
y_test       = test[target]

print(f"Train: {len(train)} rows | Test: {len(test)} rows")
print(f"Cutoff date: {cutoff.date()}")

# ── 4. random forest without deployment ──────────────────────────────────────
rf_base = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_base.fit(X_train_base, y_train)
rf_base_preds = rf_base.predict(X_test_base)

print("\nRF (no deployment):")
print(f"  MAE: {mean_absolute_error(y_test, rf_base_preds):.4f}")
print(f"  R2:  {r2_score(y_test, rf_base_preds):.4f}")

# ── 5. random forest with deployment ─────────────────────────────────────────
rf_dep = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_dep.fit(X_train_dep, y_train)
rf_dep_preds = rf_dep.predict(X_test_dep)

print("\nRF (with deployment):")
print(f"  MAE: {mean_absolute_error(y_test, rf_dep_preds):.4f}")
print(f"  R2:  {r2_score(y_test, rf_dep_preds):.4f}")

# ── 6. xgboost without deployment ────────────────────────────────────────────
xgb_base = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                              random_state=42, n_jobs=-1, verbosity=0)
xgb_base.fit(X_train_base, y_train)
xgb_base_preds = xgb_base.predict(X_test_base)

print("\nXGB (no deployment):")
print(f"  MAE: {mean_absolute_error(y_test, xgb_base_preds):.4f}")
print(f"  R2:  {r2_score(y_test, xgb_base_preds):.4f}")

# ── 7. xgboost with deployment ────────────────────────────────────────────────
xgb_dep = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                             random_state=42, n_jobs=-1, verbosity=0)
xgb_dep.fit(X_train_dep, y_train)
xgb_dep_preds = xgb_dep.predict(X_test_dep)

print("\nXGB (with deployment):")
print(f"  MAE: {mean_absolute_error(y_test, xgb_dep_preds):.4f}")
print(f"  R2:  {r2_score(y_test, xgb_dep_preds):.4f}")

# ── 8. results summary ────────────────────────────────────────────────────────
results = pd.DataFrame({
    'Model': ['RF (no deployment)', 'RF (with deployment)', 
              'XGB (no deployment)', 'XGB (with deployment)'],
    'MAE':   [mean_absolute_error(y_test, rf_base_preds),
              mean_absolute_error(y_test, rf_dep_preds),
              mean_absolute_error(y_test, xgb_base_preds),
              mean_absolute_error(y_test, xgb_dep_preds)],
    'R2':    [r2_score(y_test, rf_base_preds),
              r2_score(y_test, rf_dep_preds),
              r2_score(y_test, xgb_base_preds),
              r2_score(y_test, xgb_dep_preds)]
}).round(4)

print("\n", results.to_string(index=False))

# ── 9. feature importance plots ───────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, model, title, features in [
    (axes[0], rf_dep,  'RF feature importance (with deployment)',  deployment_features),
    (axes[1], xgb_dep, 'XGB feature importance (with deployment)', deployment_features)
]:
    importances = pd.Series(model.feature_importances_, index=features).sort_values()
    importances.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('Importance')
    ax.axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
#what's actually driving predictions
importances_xgb = pd.Series(
    xgb_dep.feature_importances_, 
    index=deployment_features
).sort_values(ascending=False)

print("XGB feature importances (with deployment):")
print(importances_xgb.round(4).to_string())

# variance in the target
print("\nCrime count distribution:")
print(daily_grid['crime_count'].describe().round(2))

#date range
print("\nDate range:", daily_grid['ARREST_DATE'].min(), "→", daily_grid['ARREST_DATE'].max())

In [ ]:
print(daily_grid['ARREST_DATE'].value_counts().sort_index().head(20))
print("\nUnique dates:", daily_grid['ARREST_DATE'].nunique())
print("Unique grid cells:", daily_grid['grid_id'].nunique())

FOR MODELING TARGET VARIABLE WILL BE CRIME COUNT WITH FEATURES LIKE DEPLOYED FORCES, DAY OF WEEK, MONTH, OTHER TEMPORAL, BOROUGH, PRECINT, LAGGED CRIME (YESTERDAY, 7 DAYS, 30 DAYS)!! RUN A RANDOM FOREST/XGBOOST WITH DEPLOYED FORCES AND WITHOUT
- MAYBE INCLUDE A CONSTRAINT THAT HAS A MINIMUM DEPLOYMENT FLOOR OR PENALTY FOR OVER-CONCENTRATION